In [0]:
displayHTML("""
<div style="text-align:center; width:100%;">

<h1><b>UNIVERSIDAD NACIONAL TORIBIO RODRÍGUEZ DE MENDOZA DE AMAZONAS</b></h1>

<img src="https://upload.wikimedia.org/wikipedia/commons/9/95/Logo-2016-solo-ok.png" width="150">

<h3><b>FACULTAD DE INGENIERÍA ZOOTECNISTA, BIOTECNOLOGÍA, AGRONEGOCIOS Y CIENCIA DE DATOS</b></h3>

<h3><b>ESCUELA PROFESIONAL DE INGENIERÍA EN CIENCIA DE DATOS E INTELIGENCIA ARTIFICIAL</b></h3>

<hr>

<h2><b>Análisis de Entidades y Personajes Literarios mediante NLP en Obras Narrativas</b></h2>

<hr>

<table style="margin-left:auto; margin-right:auto;">
  <tr>
    <td><b>Alumno:</b></td>
    <td>López Llaja, Abel Ricardo</td>
  </tr>
  <tr>
    <td><b>Profesor:</b></td>
    <td>Mamani Bustamante, Juan Pablo</td>
  </tr>
  <tr>
    <td><b>Curso:</b></td>
    <td>Procesamiento del Lenguaje Natural</td>
  </tr>
  <tr>
    <td><b>Ciclo:</b></td>
    <td>V</td>
  </tr>
</table>

<hr>

<h3><b>CHACHAPOYAS – PERÚ</b></h3>
<h3><b>2026-I</b></h3>

</div>
""")

Alumno:,"López Llaja, Abel Ricardo"
Profesor:,"Mamani Bustamante, Juan Pablo"
Curso:,Procesamiento del Lenguaje Natural
Ciclo:,V


# 1. Introducción
En este notebook aplican técnicas de Procesamiento de Lenguaje Natural a dos obras literarias: **Conversación en La Catedral** de Mario Vargas Llosa y **Comentarios reales de los incas** del Inca Garcilaso de la Vega. El propósito es extraer entidades relevantes como personajes, lugares y escenarios, y posteriormente analizar relaciones entre personajes mediante redes de coocurrencia.

# 2. Objetivos del análisis
**Objetivo general:**

- Aplicar técnicas de NLP para extraer entidades literarias y analizar relaciones entre personajes en dos obras de la literatura peruana.

**Objetivos específicos:**
- Cargar y organizar los textos por unidad narrativa.
- Limpiar y segmentar los textos para su procesamiento.
- Extraer entidades nombradas usando Flair.
- Clasificar entidades en personajes, lugares y otras categorías.
- Construir redes de coocurrencia entre personajes.
- Calcular métricas de centralidad.
- Visualizar redes mediante PyVis.

# 3. Descripción del corpus

El corpus utilizado en este trabajo está compuesto por dos obras literarias peruanas en formato `.txt`:

| Código de obra | Título | Autor | Tipo de texto |
|---|---|---|---|
| CELC_MVLL | Conversación en La Catedral | Mario Vargas Llosa | Novela |
| CRDLI_IGDLV | Comentarios reales de los incas | Inca Garcilaso de la Vega | Crónica histórica |

Ambos textos fueron obtenidos previamente a partir de archivos de extensión `.epub` y convertidos a texto plano. Para simplificar el flujo de procesamiento, se trabajó con cada obra como un libro completo, en lugar de analizar sus divisiones internas por partes, libros o capítulos.

Los textos fueron cargados desde archivos alojados en [GitHub](https://github.com/elable947/proyecto_nlp_libros.git) mediante URLs en formato `raw`. Posteriormente, cada libro fue segmentado automáticamente en bloques de texto para facilitar el procesamiento con herramientas de NLP.

El análisis se centra en:

- Extracción de entidades nombradas.
- Identificación de personajes.
- Identificación de lugares y escenarios.
- Construcción de redes de coocurrencia entre personajes.
- Cálculo de métricas de centralidad.
- Visualización interactiva de relaciones mediante PyVis.

# 4. Importación de librerias

In [0]:
%pip install flair spacy networkx pyvis

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Instalación del modelo pequeño de spaCy para español.
# Se instala desde el wheel oficial para asegurar compatibilidad con spaCy 3.8.
%pip install https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 110.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#dbutils.library.restartPython() # reiniciamos el kernel para poder usar las librerías instaladas

In [0]:
import os
import re
from pathlib import Path
from collections import Counter
from itertools import combinations

# Descarga y manipulación de datos
import requests
import pandas as pd

# Procesamiento de lenguaje natural
import spacy
from flair.data import Sentence
from flair.models import SequenceTagger

# Redes y visualización
import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt

# 5. Configuración de rutas y parámetros

In [0]:
URL_CELC = "https://raw.githubusercontent.com/elable947/proyecto_nlp_libros/refs/heads/main/data/txt/CELC_MVLL/CELC_MVLL_libro_completo.txt"

URL_CRDLI = "https://raw.githubusercontent.com/elable947/proyecto_nlp_libros/refs/heads/main/data/txt/CRDLI_IGDLV/CRDLI_IGDLV_libro_completo.txt"

# Parámetros principales del análisis.
MAX_PALABRAS_BLOQUE = 700      # Tamaño aproximado de cada bloque de texto.
UMBRAL_SCORE_NER = 0.80        # Umbral reservado para filtrar entidades si se desea mayor precisión.
MIN_PESO_RED = 2               # Peso mínimo para conservar una relación de coocurrencia.

OBRAS = {
    "CELC_MVLL": "Conversación en La Catedral",
    "CRDLI_IGDLV": "Comentarios reales de los incas"
}

RUTA_SALIDA = "/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros" # para guardar resultados
RUTA_GRAFOS = f"{RUTA_SALIDA}/grafos"

# Se crean las carpetas de salida si el volumen ya existe.
Path(RUTA_SALIDA).mkdir(parents=True, exist_ok=True)
Path(RUTA_GRAFOS).mkdir(parents=True, exist_ok=True)

print("Ruta de salida configurada:", RUTA_SALIDA)
print("Ruta para grafos configurada:", RUTA_GRAFOS)

Ruta de salida configurada: /Volumes/workspace/default/mi_volumen/proyecto_nlp_libros
Ruta para grafos configurada: /Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/grafos


# 6. Carga de libros completos

In [0]:
def cargar_texto_desde_url(url): # funcion para cargar los libros desde github
    response = requests.get(url)
    response.raise_for_status()
    return response.text

In [0]:
contenido_celc = cargar_texto_desde_url(URL_CELC)
contenido_crdli = cargar_texto_desde_url(URL_CRDLI)
print(contenido_celc[:500])

Conversación en La Catedral

Mario Vargas Llosa

UNO
I


Desde la puerta de
La Crónica
 Santiago mira la avenida Tacna, sin amor: automóviles, edificios desiguales y descoloridos, esqueletos de avisos luminosos flotando en la neblina, el mediodía gris. ¿En qué momento se había jodido el Perú? Los canillitas merodean entre los vehículos detenidos por el semáforo de Wilson voceando los diarios de la tarde y él echa a andar, despacio, hacia la Colmena. Las manos en los bolsillos, cabizbajo, va esco


In [0]:
print(contenido_crdli[:500])

Comentarios Reales de los Incas

Inca Garcilaso de la Vega

LIBRO PRIMERO


donde se trata el descubrimiento del Nuevo Mundo, la deducción del

 nombre Perú, la idolatría y manera de vivir antes de los Reyes Incas,

el origen de ellos, la vida del primer Inca y lo que hizo con sus

vasallos, y la significación de los nombres reales.


Contiene veinte y seis capítulos.

Capítulo I: Si hay muchos mundos. Trata de las cinco Zonas.


Habiendo de tratar del Nuevo Mundo, o de la mejor y más principal



In [0]:
df_textos = pd.DataFrame([
    {
        "obra": "CELC_MVLL",
        "titulo": "Conversación en La Catedral",
        "autor": "Mario Vargas Llosa",
        "texto": contenido_celc
    },
    {
        "obra": "CRDLI_IGDLV",
        "titulo": "Comentarios reales de los incas",
        "autor": "Inca Garcilaso de la Vega",
        "texto": contenido_crdli
    }
])

df_textos["num_caracteres"] = df_textos["texto"].str.len()
df_textos["num_palabras_aprox"] = df_textos["texto"].str.split().str.len()

display(df_textos[["obra", "titulo", "autor", "num_caracteres", "num_palabras_aprox"]])

obra,titulo,autor,num_caracteres,num_palabras_aprox
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1186153,208831
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1513463,266275


# 7. Limpieza y normalización básica

Antes de aplicar NLP, se normalizan saltos de línea, espacios repetidos y caracteres invisibles. Esta limpieza no elimina puntuación ni tildes, porque esos elementos pueden ayudar al reconocimiento de entidades y a conservar mejor el contexto literario.

In [0]:
def limpiar_texto(texto):
    texto = texto.replace("\ufeff", "")
    texto = texto.replace("\r\n", "\n")
    texto = texto.replace("\r", "\n")
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = texto.strip()
    return texto

df_textos["texto_limpio"] = df_textos["texto"].apply(limpiar_texto)

# 8. Segmentación automática en bloques

Los modelos de NLP pueden ser costosos cuando se aplican sobre libros completos. Por eso, cada obra se divide en bloques de tamaño controlado. Cada bloque mantiene metadatos de obra, título, autor e identificador numérico.

En este trabajo, el bloque funciona como unidad de análisis: las entidades se extraen por bloque y las coocurrencias entre personajes se calculan cuando aparecen dentro del mismo bloque.

In [0]:
def dividir_en_bloques(texto, max_palabras=700):
    parrafos = [p.strip() for p in texto.split("\n\n") if p.strip()]
    
    bloques = []
    bloque_actual = []
    palabras_actuales = 0

    for parrafo in parrafos:
        palabras = parrafo.split()
        n_palabras = len(palabras)

        if palabras_actuales + n_palabras > max_palabras and bloque_actual:
            bloques.append("\n\n".join(bloque_actual))
            bloque_actual = []
            palabras_actuales = 0

        bloque_actual.append(parrafo)
        palabras_actuales += n_palabras

    if bloque_actual:
        bloques.append("\n\n".join(bloque_actual))

    return bloques

In [0]:
registros_bloques = []

for _, row in df_textos.iterrows():
    bloques = dividir_en_bloques(
        row["texto_limpio"],
        max_palabras=MAX_PALABRAS_BLOQUE
    )

    for i, bloque in enumerate(bloques, start=1):
        registros_bloques.append({
            "obra": row["obra"],
            "titulo": row["titulo"],
            "autor": row["autor"],
            "bloque_id": i,
            "texto_bloque": bloque,
            "num_palabras_bloque": len(bloque.split())
        })

df_bloques = pd.DataFrame(registros_bloques)

display(df_bloques.head())

obra titulo autor bloque_id texto_bloque num_palabras_bloque CELC_MVLL Conversación en La Catedral Mario Vargas Llosa 1 Conversación en La Catedral

Mario Vargas Llosa

UNO
I

Desde la puerta de
La Crónica
 Santiago mira la avenida Tacna, sin amor: automóviles, edificios desiguales y descoloridos, esqueletos de avisos luminosos flotando en la neblina, el mediodía gris. ¿En qué momento se había jodido el Perú? Los canillitas merodean entre los vehículos detenidos por el semáforo de Wilson voceando los diarios de la tarde y él echa a andar, despacio, hacia la Colmena. Las manos en los bolsillos, cabizbajo, va escoltado por transeúntes que avanzan, también, hacia la Plaza San Martín. El era como el Perú, Zavalita, se había jodido en algún momento. Piensa: ¿en cuál? Frente al Hotel Crillón un perro viene a lamerle los pies: no vayas a estar rabioso, fuera de aquí. El Perú jodido, piensa, Carlitos jodido, todos jodidos. Piensa: no hay solución. Ve una larga cola en el paradero de los colectivos a Miraflores, cruza la Plaza y ahí está Norwin, hola hermano, en una mesa del Bar Zela, siéntate Zavalita, manoseando un chilcano y haciéndose lustrar los zapatos, le invitaba un trago. No parece borracho todavía y Santiago se sienta, indica al lustrabotas que también le lustre los zapatos a él. Listo jefe, ahoritita jefe, se los dejaría como espejos, jefe.

—Siglos que no se te ve, señor editorialista —dice Norwin —. ¿Estás más contento en la página editorial que en locales?

—Se trabaja menos —alza los hombros, a lo mejor había sido ese día que el Director lo llamó, pide una Cristal helada, fría reemplazar a Orgambide, Zavalita?, él había estado en la Universidad y podría escribir editoriales ¿no, Zavalita? Piensa: ahí me jodí —. Vengo temprano, me da mi tema, me tapo la nariz y en dos o tres horas, listo, jalo la cadena y ya está.

—Yo no haría editoriales ni por todo el oro del mundo —dice Norwin —. Estás lejos de la noticia y el periodismo es noticia, Zavalita, Convéncete. Me moriré en policiales, nomás. A propósito ¿se murió Carlitos?

—Sigue en la clínica, pero le darán de alta pronto —dice Santiago —. Jura que va a dejar el trago esta vez.

—¿Cierto que una noche al acostarse vio cucarachas y arañas? —dice Norwin.

—Levantó la sábana y se le vinieron encima miles de tarántulas, de ratones —dijo Santiago —. Salió calato a la calle dando gritos.

Norwin se ríe y Santiago cierra los ojos: las casas de Chorrillos son cubos con rejas, cuevas agrietadas por temblores, en el interior hormiguean cachivaches y polvorientas viejecillas pútridas, en zapatillas, con varices. Una figurilla corre entre los cubos, sus alaridos estremecen la aceitosa madrugada y enfurecen a las hormigas, alacranes y escorpiones que la persiguen. La consolación por el alcohol; piensa, contra la muerte lenta los diablos azules. Estaba bien, Carlitos, uno se defendía del Perú como podía.

—El día menos pensado yo también me voy a encontrar a los bichitos —Norwin contempla su chilcano con curiosidad, sonríe a medias —. Pero no hay periodista abstemio, Zavalita. El trago inspira, convéncete.

El lustrabotas ha terminado con Norwin y ahora embetuna los zapatos de Santiago, silbando. ¿Cómo iban las cosas por
Última Hora
?, ¿qué se contaban esos bandoleros? Se quejaban de la ingratitud, Zavalita, que viniera alguna vez a visitarlos, como antes. O sea que ahora tenías un montón de tiempo libre, Zavalita, ¿trabajabas en otro sitio?

—Leo, duermo siestas —dice Santiago —. Quizá me matricule otra vez en Derecho.

—Te alejas de la noticia y ya quieres un título —Norwin lo mira apenado —. La página editorial es el fin, Zavalita. Te recibirás de abogado, dejarás el periodismo. Ya te estoy viendo hecho un burgués.

—Acabo de cumplir treinta años —dice Santiago —. Tarde para volverme un burgués.

—¿Treinta, nada más? —Norwin queda pensativo —. Yo treintaiséis y parezco tu padre. La página policial lo muele a uno, convéncete.

Caras masculinas, ojos opacos y derrotados sobre las mesa

In [0]:
display(df_bloques[333:].head())

obra,titulo,autor,bloque_id,texto_bloque,num_palabras_bloque
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,"Comentarios Reales de los Incas Inca Garcilaso de la Vega LIBRO PRIMERO donde se trata el descubrimiento del Nuevo Mundo, la deducción del nombre Perú, la idolatría y manera de vivir antes de los Reyes Incas, el origen de ellos, la vida del primer Inca y lo que hizo con sus vasallos, y la significación de los nombres reales. Contiene veinte y seis capítulos. Capítulo I: Si hay muchos mundos. Trata de las cinco Zonas. Habiendo de tratar del Nuevo Mundo, o de la mejor y más principal parte suya, que son los reinos y provincias del Imperio llamado Perú, de cuyas antiguallas y origen de sus Reyes pretendemos escribir, parece que fuera justo, conforme a la común costumbre de los escritores, tratar aquí al principio si el mundo es uno sólo o si hay muchos mundos; si es llano o redondo, y si también lo es el cielo redondo o llano; si es habitable toda la tierra o no más de las zonas templadas; si hay paso de una templada a la otra; si hay antípodas y cuáles son de cuáles, y otras cosas semejantes que los antiguos filósofos muy larga y curiosamente trataron y los modernos no dejan de platicar y escribir, siguiendo cada cual opinión que más le agrada. Mas porque no es aqueste mi principal intento ni las fuerzas de un indio pueden presumir tanto, y también porque la experiencia, después que se descubrió lo que llaman Nuevo Mundo, nos ha desengañado de la mayor parte de estas dudas, pasaremos brevemente por ellas, por ir a otra parte, a cuyos términos finales temo no llegar. Mas confiado en la infinita misericordia, digo que a lo primero se podrá afirmar que no hay más que un mundo, y aunque llamarnos Mundo Viejo y Mundo Nuevo, es por haberse descubierto aquél nuevamente para nosotros, y no porque sean dos, sino todo uno. Y a los que todavía imaginaren que hay muchos mundos, no hay para qué responderles, sino que se estén en sus heréticas imaginaciones hasta que en el infierno se desengañen de ellas. Y a los que dudan, si hay alguno que lo dude, si es llano o redondo se podrá satisfacer con el testimonio de los que han dado vuelta a todo él o a la mayor parte, como los de la nao Victoria y otros que después acá le han rodeado. Y a lo del cielo, si también es llano o redondo, se podrá responder con las palabras del Real Profeta: Extendens c&lum, sicut pellem en las cuales nos quiso mostrar la forma y hechura de la obra, dando la una por ejemplo de la otra, diciendo: «Que extendiste el cielo así como la piel», esto es, cubriendo con el cielo este gran cuerpo de los cuatro elementos en redondo, así como cubriste con la piel en redondo el cuerpo del animal, no solamente lo principal de él, mas también todas sus partes, por pequeñas que sean. A los que afirman que de las cinco partes del mundo que llaman zonas no son habitables más de las dos templadas, y que la del medio por su [e]xcesivo calor y las dos de los cabos por el demasiado frío son inhabitables, y que de la una zona habitable no se puede pasar a la otra habitable por el calor demasiado que hay en medio, puedo afirmar, demás de lo que todos saben, que yo nací en la tórrida zona, que es en el Cozco, y me crié en ella hasta los veinte años, y he estado en la otra zona templada de la otra parte del Trópico de Capricornio, a la parte del sur, en los últimos términos de los Charcas, que son los Chichas, y, para venir a esta otra templada de la parte del norte, donde escribo esto, pasé por la tórrida zona y la atravesé toda y estuve tres días naturales debajo de la línea equinoccial, donde dicen que pasa perpendicularmente, que es en el cabo de Pasau por todo lo cual digo que es habitable la tórrida también como las templadas. De las zonas frías quisiera poder decir por vista de ojos como de las otras tres. Remítame a",697
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,2,"los que saben de ellas más que yo. A los que dicen que por su 

In [0]:
display(
    df_bloques.groupby("obra").agg(
        cantidad_bloques=("bloque_id", "count"),
        promedio_palabras=("num_palabras_bloque", "mean"),
        min_palabras=("num_palabras_bloque", "min"),
        max_palabras=("num_palabras_bloque", "max")
    ).reset_index()
)

obra,cantidad_bloques,promedio_palabras,min_palabras,max_palabras
CELC_MVLL,333,627.1201201201201,1,1035
CRDLI_IGDLV,384,693.4244791666666,599,700


Aunque se definió un máximo de `700` palabras por bloque, este valor funciona como un límite aproximado. 
La función prioriza no cortar párrafos completos; por ello, si un párrafo individual supera las 700 palabras, 
ese párrafo se conserva como un solo bloque. Esto explica que el máximo observado pueda ser mayor al límite configurado.

El mínimo `1` probablemente viene de un párrafo muy corto, como un título, número de capítulo, separador o texto suelto.

# 9. Procesamiento lingüístico auxiliar con spaCy
Se carga el modelo en español de spaCy. En este notebook, spaCy se utiliza como apoyo lingüístico para tokenización y estructura del texto, mientras que la extracción principal de entidades se realiza con Flair.

Entonces con spaCy:
- tokenización
- segmentación
- limpieza lingüística
- detección de oraciones
- posible lematización
- posible análisis gramatical

La opción `disable=["ner"]` evita ejecutar el NER de spaCy y reduce el costo de procesamiento, ya que el reconocimiento de entidades se hará en la siguiente sección con Flair.

In [0]:
nlp = spacy.load("es_core_news_sm")

In [0]:
docs_spacy = []

for doc in nlp.pipe(
    df_bloques["texto_bloque"],
    batch_size=16, # lo podemos cambiar por 8
    disable=["ner"]
):
    docs_spacy.append(doc)

df_bloques["doc_spacy"] = docs_spacy

In [0]:
df_bloques.head(10)

,obra,titulo,autor,bloque_id,texto_bloque,num_palabras_bloque,doc_spacy
0,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Conversación en La Catedral\n\nMario Vargas Ll...,688,"(Conversación, en, La, Catedral, \n\n, Mario, ..."
1,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,2,—¿Hasta cuándo va a seguir la campaña de\nLa C...,241,"(—, ¿, Hasta, cuándo, va, a, seguir, la, campa..."
2,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,3,"Todavía hay embotellamientos en el centro, per...",693,"(Todavía, hay, embotellamientos, en, el, centr..."
3,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,4,"Almuerzan sin hablar, en la mesita pegada a la...",555,"(Almuerzan, sin, hablar, ,, en, la, mesita, pe..."
4,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,5,"Que no se pusiera así, amigo periodista, no er...",698,"(Que, no, se, pusiera, así, ,, amigo, periodis..."
5,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,6,"No era él, todos los negros se parecían, no po...",699,"(No, era, él, ,, todos, los, negros, se, parec..."
6,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,7,"¿Y la niña Teté?, y las manazas van y vienen, ...",490,"(¿, Y, la, niña, Teté, ?, ,, y, las, manazas, ..."
7,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,8,"Huele a sudor, ají y cebolla, a orines y basur...",868,"(Huele, a, sudor, ,, ají, y, cebolla, ,, a, or..."
8,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,9,"—Menos mal que le alcanzó la plata, niño. ¿De ...",698,"(—, Menos, mal, que, le, alcanzó, la, plata, ,..."
9,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,10,"—Ya está —solloza Santiago, inclinándose, acar...",647,"(—, Ya, está, —, solloza, Santiago, ,, incliná..."


In [0]:
df_bloques[333:].head(10)

,obra,titulo,autor,bloque_id,texto_bloque,num_palabras_bloque,doc_spacy
333,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,Comentarios Reales de los Incas\n\nInca Garcil...,697,"(Comentarios, Reales, de, los, Incas, \n\n, In..."
334,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,2,los que saben de ellas más que yo. A los que d...,696,"(los, que, saben, de, ellas, más, que, yo, ., ..."
335,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,3,llevar tantos ¿cómo no quedaron acá de los que...,694,"(llevar, tantos, ¿, cómo, no, quedaron, acá, d..."
336,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,4,Colón les regaló no pudieron volver en sí y mu...,699,"(Colón, les, regaló, no, pudieron, volver, en,..."
337,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,5,"Pues hemos de tratar del Perú, será bien digam...",677,"(Pues, hemos, de, tratar, del, Perú, ,, será, ..."
338,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,6,"de aquella tierra, por que si tomaron el nombr...",693,"(de, aquella, tierra, ,, por, que, si, tomaron..."
339,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,7,"los naturales Pirú, intitularon toda esta tier...",688,"(los, naturales, Pirú, ,, intitularon, toda, e..."
340,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,8,tan destrozados que falta lo más y mejor; hízo...,699,"(tan, destrozados, que, falta, lo, más, y, mej..."
341,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,9,aquella tierra que corre desde la equinoccial ...,690,"(aquella, tierra, que, corre, desde, la, equin..."
342,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,10,"La isla que ha por nombre la Trinidad, que est...",694,"(La, isla, que, ha, por, nombre, la, Trinidad,..."


In [0]:
doc_ejemplo = df_bloques.loc[0, "doc_spacy"]

analisis_tokens = []

for token in doc_ejemplo[:30]:
    analisis_tokens.append({
        "token": token.text,
        "lema": token.lemma_,
        "categoria_gramatical": token.pos_,
        "dependencia": token.dep_,
        "es_stopword": token.is_stop,
        "es_puntuacion": token.is_punct
    })

df_analisis_spacy = pd.DataFrame(analisis_tokens)

display(df_analisis_spacy)

token,lema,categoria_gramatical,dependencia,es_stopword,es_puntuacion
Conversación,Conversación,PROPN,nsubj,false,false
en,en,ADP,case,true,false
La,el,DET,det,true,false
Catedral,Catedral,PROPN,nmod,false,false
,,SPACE,dep,false,false
Mario,Mario,PROPN,flat,false,false
Vargas,Vargas,PROPN,flat,false,false
Llosa,Llosa,PROPN,flat,false,false
,,SPACE,dep,false,false
UNO,UNO,PROPN,flat,true,false


Para entender que significa cada etiqueta de la columna `categoria_gramatical` revisar la documentación de spaCy sobre [Part-of-Speech Tagging](https://spacy.io/usage/linguistic-features#pos-tagging) o [Universal POS tags](https://universaldependencies.org/u/pos/)

In [0]:
# Ejemplo de detección de oraciones con spaCy
oraciones = []

for i, sent in enumerate(doc_ejemplo.sents, start=1):
    oraciones.append({
        "oracion_id": i,
        "oracion": sent.text,
        "num_tokens": len(sent)
    })

df_oraciones_ejemplo = pd.DataFrame(oraciones)

display(df_oraciones_ejemplo.head(10))

oracion_id,oracion,num_tokens
1,"Conversación en La Catedral Mario Vargas Llosa UNO I Desde la puerta de La Crónica Santiago mira la avenida Tacna, sin amor: automóviles, edificios desiguales y descoloridos, esqueletos de avisos luminosos flotando en la neblina, el mediodía gris.",50
2,¿En qué momento se había jodido el Perú?,10
3,"Los canillitas merodean entre los vehículos detenidos por el semáforo de Wilson voceando los diarios de la tarde y él echa a andar, despacio, hacia la Colmena.",30
4,"Las manos en los bolsillos, cabizbajo, va escoltado por transeúntes que avanzan, también, hacia la Plaza San Martín.",23
5,"El era como el Perú, Zavalita, se había jodido en algún momento.",15
6,Piensa: ¿en cuál?,6
7,"Frente al Hotel Crillón un perro viene a lamerle los pies: no vayas a estar rabioso, fuera de aquí.",22
8,"El Perú jodido, piensa, Carlitos jodido, todos jodidos.",12
9,Piensa: no hay solución.,6
10,"Ve una larga cola en el paradero de los colectivos a Miraflores, cruza la Plaza y ahí está Norwin, hola hermano, en una mesa del Bar Zela, siéntate Zavalita, manoseando un chilcano y haciéndose lustrar los zapatos, le invitaba un trago.",48


In [0]:
def obtener_metricas_spacy(doc):
    tokens = [token for token in doc if not token.is_space]
    tokens_alpha = [token for token in tokens if token.is_alpha]
    oraciones = list(doc.sents)

    return {
        "num_tokens_spacy": len(tokens),
        "num_tokens_alpha": len(tokens_alpha),
        "num_oraciones": len(oraciones)
    }

metricas_spacy = df_bloques["doc_spacy"].apply(obtener_metricas_spacy)

df_metricas_spacy = pd.DataFrame(metricas_spacy.tolist())

df_bloques = pd.concat(
    [df_bloques.reset_index(drop=True), df_metricas_spacy],
    axis=1
)

display(df_bloques[[
    "obra",
    "bloque_id",
    "num_palabras_bloque",
    "num_tokens_spacy",
    "num_tokens_alpha",
    "num_oraciones"
]].head())

obra,bloque_id,num_palabras_bloque,num_tokens_spacy,num_tokens_alpha,num_oraciones
CELC_MVLL,1,688,853,677,53
CELC_MVLL,2,241,297,236,18
CELC_MVLL,3,693,859,684,53
CELC_MVLL,4,555,659,548,29
CELC_MVLL,5,698,870,689,50


In [0]:
display(df_bloques[[
    "obra",
    "bloque_id",
    "num_palabras_bloque",
    "num_tokens_spacy",
    "num_tokens_alpha",
    "num_oraciones"
]][333:].head())

obra,bloque_id,num_palabras_bloque,num_tokens_spacy,num_tokens_alpha,num_oraciones
CRDLI_IGDLV,1,697,777,695,13
CRDLI_IGDLV,2,696,789,696,14
CRDLI_IGDLV,3,694,769,694,15
CRDLI_IGDLV,4,699,784,697,10
CRDLI_IGDLV,5,677,762,677,14


# 10. Extracción de entidades nombradas con Flair

[Flair](https://github.com/flairNLP/flair.git) se utiliza para detectar entidades nombradas en español. Cada entidad extraída incluye el texto detectado, el tipo de entidad y un score de confianza.

Los tipos más relevantes para el análisis son:

- `PER`: personas, personajes o figuras mencionadas.
- `LOC`: lugares, espacios o referencias geográficas.
- `ORG`: organizaciones o instituciones.
- `MISC`: otras entidades diversas.

El resultado de esta sección será `df_entidades`, una tabla larga donde cada fila representa una entidad detectada en un bloque específico.

In [0]:
tagger = SequenceTagger.load("flair/ner-spanish-large")

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-05-07 17:16:41,526 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-LOC, S-ORG, B-PER, I-PER, E-PER, S-MISC, B-ORG, E-ORG, S-PER, I-ORG, B-LOC, E-LOC, B-MISC, E-MISC, I-MISC, I-LOC, <START>, <STOP>


In [0]:
def extraer_entidades_flair(texto):
    sentence = Sentence(texto)
    tagger.predict(sentence)

    entidades = []

    for ent in sentence.get_spans("ner"):
        entidades.append({
            "entidad": ent.text,
            "tipo": ent.tag,
            "score": ent.score
        })

    return entidades

In [0]:
registros_entidades = []

for i, row in df_bloques.iterrows():
    entidades = extraer_entidades_flair(row["texto_bloque"])

    for ent in entidades:
        registros_entidades.append({
            "obra": row["obra"],
            "titulo": row["titulo"],
            "autor": row["autor"],
            "bloque_id": row["bloque_id"],
            "entidad": ent["entidad"],
            "tipo": ent["tipo"],
            "score": ent["score"]
        })

    if (i + 1) % 20 == 0:
        print(f"Bloques procesados: {i + 1}/{len(df_bloques)}")

df_entidades = pd.DataFrame(registros_entidades)

display(df_entidades.head())

Bloques procesados: 20/717
Bloques procesados: 40/717
Bloques procesados: 60/717
Bloques procesados: 80/717
Bloques procesados: 100/717
Bloques procesados: 120/717
Bloques procesados: 140/717
Bloques procesados: 160/717
Bloques procesados: 180/717
Bloques procesados: 200/717
Bloques procesados: 220/717
Bloques procesados: 240/717
Bloques procesados: 260/717
Bloques procesados: 280/717
Bloques procesados: 300/717
Bloques procesados: 320/717
Bloques procesados: 340/717
Bloques procesados: 360/717
Bloques procesados: 380/717
Bloques procesados: 400/717
Bloques procesados: 420/717
Bloques procesados: 440/717
Bloques procesados: 460/717
Bloques procesados: 480/717
Bloques procesados: 500/717
Bloques procesados: 520/717
Bloques procesados: 540/717
Bloques procesados: 560/717
Bloques procesados: 580/717
Bloques procesados: 600/717
Bloques procesados: 620/717
Bloques procesados: 640/717
Bloques procesados: 660/717
Bloques procesados: 680/717
Bloques procesados: 700/717


obra,titulo,autor,bloque_id,entidad,tipo,score
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Catedral,LOC,0.9997674822807312
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Mario Vargas Llosa,PER,0.9998625914255778
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Crónica,LOC,0.9999179244041443
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Santiago,PER,0.998166024684906
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,avenida Tacna,LOC,0.9997281134128571


In [0]:
df_entidades.sort_values('score', ascending=False).head(10)

,obra,titulo,autor,bloque_id,entidad,tipo,score
18542,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,257,Madrid,LOC,0.999989
20663,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,354,España,LOC,0.999989
20664,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,354,Andalucía,LOC,0.999987
20722,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,357,Bilbao,LOC,0.999987
7819,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,203,Arequipa,LOC,0.999987
20353,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,341,España,LOC,0.999987
3877,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,94,Arequipa,LOC,0.999987
20822,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,362,España,LOC,0.999986
19939,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,322,Perú,LOC,0.999986
18362,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,252,España,LOC,0.999986


In [0]:
df_entidades.sort_values('score', ascending=True).head(10)

,obra,titulo,autor,bloque_id,entidad,tipo,score
17507,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,214,chúcam,PER,0.156536
18283,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,247,Sol,MISC,0.295542
16433,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,168,Sol,ORG,0.296423
18506,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,256,Incas,PER,0.314076
15264,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,107,Incas,PER,0.321484
19121,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,278,Señor,ORG,0.325116
14023,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,53,Incas,PER,0.332770
16516,CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,171,Señor,MISC,0.333013
771,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,23,Flor,ORG,0.335428
10331,CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,276,Hilario,PER,0.344650


Esta es la parte que más tiempo toma en ejecución ya que se analizan todos los bloques de ambos libros, en pocas palabras se analizan ambos libros completos

# 11. Normalización y depuración de entidades

Las entidades extraídas por Flair pueden presentar variaciones de escritura debido al contexto narrativo, uso de mayúsculas/minúsculas, signos de puntuación o apodos literarios.

Para reducir duplicados y mejorar la coherencia del análisis, en esta etapa se realiza:

- limpieza de espacios y caracteres residuales;
- normalización textual;
- unificación en minúsculas para comparación interna;
- agrupación de alias y apodos mediante un diccionario manual;
- construcción de una forma canónica por personaje.

Este proceso es especialmente importante en *Conversación en La Catedral*, donde varios personajes son mencionados mediante apodos, sobrenombres o variantes contextuales.

In [0]:
df_entidades_limpio = df_entidades.copy()
display(df_entidades_limpio.head())

obra,titulo,autor,bloque_id,entidad,tipo,score
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Catedral,LOC,0.9997674822807312
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Mario Vargas Llosa,PER,0.9998625914255778
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Crónica,LOC,0.9999179244041443
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Santiago,PER,0.998166024684906
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,avenida Tacna,LOC,0.9997281134128571


In [0]:
import unicodedata

# Función de normalización textual

def normalizar_texto_entidad(texto):

    """
    Normaliza entidades para comparación interna.

    Esta función:
    - conserva acentos;
    - convierte a minúsculas;
    - elimina espacios repetidos;
    - elimina signos laterales.
    """

    if pd.isna(texto):
        return texto

    texto = str(texto)

    # Unicode consistente
    texto = unicodedata.normalize("NFKC", texto)

    # Minúsculas SOLO para comparación
    texto = texto.lower()

    # Espacios repetidos
    texto = re.sub(r"\s+", " ", texto)

    # Eliminar signos laterales
    texto = texto.strip(".,;:¡!¿?()[]{}\"'")

    return texto.strip()

In [0]:
# Crear columna normalizada interna

df_entidades_limpio["entidad_norm"] = (
    df_entidades_limpio["entidad"]
    .apply(normalizar_texto_entidad)
)

# Normalizar etiquetas
df_entidades_limpio["tipo"] = (
    df_entidades_limpio["tipo"]
    .astype(str)
    .str.upper()
)

display(
    df_entidades_limpio[
        ["entidad", "entidad_norm", "tipo"]
    ].head(20)
)

entidad,entidad_norm,tipo
La Catedral,la catedral,LOC
Mario Vargas Llosa,mario vargas llosa,PER
La Crónica,la crónica,LOC
Santiago,santiago,PER
avenida Tacna,avenida tacna,LOC
Perú,perú,LOC
Wilson,wilson,PER
Colmena,colmena,LOC
Plaza San Martín,plaza san martín,LOC
Perú,perú,LOC


In [0]:
df_entidades_limpio = df_entidades_limpio[
    df_entidades_limpio["entidad_norm"].str.len() > 2
].copy()

df_entidades_limpio = df_entidades_limpio[
    ~df_entidades_limpio["entidad_norm"].str.isnumeric()
].copy()

display(df_entidades_limpio.head())

obra,titulo,autor,bloque_id,entidad,tipo,score,entidad_norm
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Catedral,LOC,0.9997674822807312,la catedral
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Mario Vargas Llosa,PER,0.9998625914255778,mario vargas llosa
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Crónica,LOC,0.9999179244041443,la crónica
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Santiago,PER,0.998166024684906,santiago
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,avenida Tacna,LOC,0.9997281134128571,avenida tacna


In [0]:
# Diccionario de alias para Conversación en La Catedral

diccionario_alias_personajes = {

    "santiago": "Santiago Zavala",
    "zavalita": "Santiago Zavala",
    "santiago zavala": "Santiago Zavala",
    "santiaguito": "Santiago Zavala",
    "el flaco": "Santiago Zavala",

    "don fermín": "Fermín Zavala",
    "fermín": "Fermín Zavala",
    "fermín zavala": "Fermín Zavala",
    "bola de oro": "Fermín Zavala",

    "cayo": "Cayo Bermúdez",
    "cayo bermúdez": "Cayo Bermúdez",
    "bermúdez": "Cayo Bermúdez",
    "don cayo": "Cayo Bermúdez",
    "cayo mierda": "Cayo Bermúdez",

    "hortensia": "Hortensia",
    "la musa": "Hortensia",

    "ambrosio": "Ambrosio Pardo",
    "ambrosio pardo": "Ambrosio Pardo",
    "el zambo ambrosio": "Ambrosio Pardo",
    "zambo ambrosio": "Ambrosio Pardo",
    "el zambo": "Ambrosio Pardo",

    "amalia": "Amalia Landa",
    "amalita": "Amalia Landa",
    "amalia landa": "Amalia Landa",

    "queta": "Queta",
    "la queta": "Queta",

    "el chispas": "Chispas Zavala",
    "chispas": "Chispas Zavala",
    "teté": "Teté Zavala",
    "la teté": "Teté Zavala",

    "popeye": "Popeye Arévalo",
    "popeye arévalo": "Popeye Arévalo",
    "arévalo": "Popeye Arévalo",
    "aída": "Aída",
    "jacobo": "Jacobo",
    "ana": "Ana",

    "trifulcas": "Trifulcas",
    "ludovico": "Ludovico",
    "hipólito": "Hipólito",
    "lozano": "Coronel Lozano"
}

In [0]:
def resolver_alias_personajes(row):

    entidad_norm = row["entidad_norm"]
    obra = row["obra"]

    # Alias solo para Conversación en La Catedral
    if obra == "CELC_MVLL":

        return diccionario_alias_personajes.get(
            entidad_norm,
            row["entidad"]
        )

    # Otras obras conservan entidad original
    return row["entidad"]

In [0]:
df_entidades_limpio["entidad_canonica"] = (
    df_entidades_limpio.apply(
        resolver_alias_personajes,
        axis=1
    )
)

display(
    df_entidades_limpio[
        ["entidad", "entidad_norm", "entidad_canonica"]
    ].head(20)
)

entidad,entidad_norm,entidad_canonica
La Catedral,la catedral,La Catedral
Mario Vargas Llosa,mario vargas llosa,Mario Vargas Llosa
La Crónica,la crónica,La Crónica
Santiago,santiago,Santiago Zavala
avenida Tacna,avenida tacna,avenida Tacna
Perú,perú,Perú
Wilson,wilson,Wilson
Colmena,colmena,Colmena
Plaza San Martín,plaza san martín,Plaza San Martín
Perú,perú,Perú


In [0]:
df_entidades_limpio["entidad_canonica"] = (
    df_entidades_limpio["entidad_canonica"]
    .astype(str)
    .str.strip()
)

In [0]:
print("Total de entidades originales:", len(df_entidades))
print("Total de entidades depuradas:", len(df_entidades_limpio))
print("Tipos de entidades encontrados:", df_entidades_limpio["tipo"].unique())
print("Obras analizadas:", df_entidades_limpio["obra"].unique())

Total de entidades originales: 21493
Total de entidades depuradas: 21468
Tipos de entidades encontrados: ['LOC' 'PER' 'MISC' 'ORG']
Obras analizadas: ['CELC_MVLL' 'CRDLI_IGDLV']


**Resultado esperado:** después de la limpieza debe mantenerse una cantidad suficiente de entidades para el análisis, pero con menos ruido textual. Si se observan demasiados falsos positivos, se puede activar el filtro por `UMBRAL_SCORE_NER`.


# 12. Análisis exploratorio de entidades

Esta sección resume las entidades en detectadas `df_entidades` mediante tablas de frecuencia, distribución por tipo, comparación entre obras y densidad por bloque.

El objetivo es obtener una primera lectura cuantitativa: qué tipos de entidades predominan, cuáles son las entidades más frecuentes y en qué partes del texto se concentran más menciones.

In [0]:
resumen_tipos = (
    df_entidades_limpio
    .groupby(["obra","tipo"])
    .size()
    .reset_index(name="cantidad")
    #.sort_values(["obra", "cantidad"], ascending=False)
)

display(resumen_tipos)

obra,tipo,cantidad
CELC_MVLL,LOC,1368
CELC_MVLL,MISC,577
CELC_MVLL,ORG,624
CELC_MVLL,PER,10162
CRDLI_IGDLV,LOC,2698
CRDLI_IGDLV,MISC,2126
CRDLI_IGDLV,ORG,596
CRDLI_IGDLV,PER,3317


In [0]:
frecuencia_entidades_obra = (
    df_entidades_limpio
    .groupby(["obra", "entidad_norm", "tipo"])
    .size()
    .reset_index(name="frecuencia")
    .sort_values(["obra", "frecuencia"], ascending=[True, False])
)

display(frecuencia_entidades_obra.head(10))

obra,entidad_norm,tipo,frecuencia
CELC_MVLL,ambrosio,PER,798
CELC_MVLL,santiago,PER,776
CELC_MVLL,amalia,PER,621
CELC_MVLL,queta,PER,407
CELC_MVLL,zavalita,PER,376
CELC_MVLL,cayo,PER,336
CELC_MVLL,ludovico,PER,321
CELC_MVLL,fermín,PER,285
CELC_MVLL,carlitos,PER,271
CELC_MVLL,chispas,PER,234


In [0]:
display(frecuencia_entidades_obra[frecuencia_entidades_obra["obra"] == "CRDLI_IGDLV"].head(10))

obra,entidad_norm,tipo,frecuencia
CRDLI_IGDLV,inca,PER,642
CRDLI_IGDLV,sol,MISC,467
CRDLI_IGDLV,incas,MISC,366
CRDLI_IGDLV,perú,LOC,363
CRDLI_IGDLV,cozco,LOC,354
CRDLI_IGDLV,rey,PER,235
CRDLI_IGDLV,incas,ORG,212
CRDLI_IGDLV,reyes,PER,189
CRDLI_IGDLV,españa,LOC,163
CRDLI_IGDLV,inca,MISC,158


In [0]:
top_n = 20

top_entidades_por_obra = (
    frecuencia_entidades_obra
    .groupby("obra", group_keys=False)
    .head(top_n)
)

display(top_entidades_por_obra)

obra,entidad_norm,tipo,frecuencia
CELC_MVLL,ambrosio,PER,798
CELC_MVLL,santiago,PER,776
CELC_MVLL,amalia,PER,621
CELC_MVLL,queta,PER,407
CELC_MVLL,zavalita,PER,376
CELC_MVLL,cayo,PER,336
CELC_MVLL,ludovico,PER,321
CELC_MVLL,fermín,PER,285
CELC_MVLL,carlitos,PER,271
CELC_MVLL,chispas,PER,234


In [0]:
comparacion_tipos_obra = (
    df_entidades_limpio
    .groupby(["obra", "tipo"])
    .size()
    .reset_index(name="cantidad")
    .sort_values(["obra", "cantidad"], ascending=[True, False])
)

display(comparacion_tipos_obra)

obra,tipo,cantidad
CELC_MVLL,PER,10162
CELC_MVLL,LOC,1368
CELC_MVLL,ORG,624
CELC_MVLL,MISC,577
CRDLI_IGDLV,PER,3317
CRDLI_IGDLV,LOC,2698
CRDLI_IGDLV,MISC,2126
CRDLI_IGDLV,ORG,596


In [0]:
comparacion_pivot = comparacion_tipos_obra.pivot_table(
    index="obra",
    columns="tipo",
    values="cantidad",
    fill_value=0
).reset_index()

display(comparacion_pivot)

obra,LOC,MISC,ORG,PER
CELC_MVLL,1368.0,577.0,624.0,10162.0
CRDLI_IGDLV,2698.0,2126.0,596.0,3317.0


In [0]:
comparacion_tipos_obra["porcentaje"] = (
    comparacion_tipos_obra
    .groupby("obra")["cantidad"]
    .transform(lambda x: x / x.sum() * 100)
)

comparacion_tipos_obra["porcentaje"] = comparacion_tipos_obra["porcentaje"].round(2)

display(comparacion_tipos_obra)

obra,tipo,cantidad,porcentaje
CELC_MVLL,PER,10162,79.82
CELC_MVLL,LOC,1368,10.75
CELC_MVLL,ORG,624,4.9
CELC_MVLL,MISC,577,4.53
CRDLI_IGDLV,PER,3317,37.96
CRDLI_IGDLV,LOC,2698,30.88
CRDLI_IGDLV,MISC,2126,24.33
CRDLI_IGDLV,ORG,596,6.82


In [0]:
import plotly.express as px

fig = px.bar(
    comparacion_tipos_obra,
    x="tipo",
    y="porcentaje",
    color="obra",
    barmode="group",
    text="porcentaje",
    title="Comparación porcentual de tipos de entidad entre obras",
    labels={
        "tipo": "Tipo de entidad",
        "porcentaje": "Porcentaje (%)",
        "obra": "Obra"
    },
    hover_data={
        "cantidad": True,
        "porcentaje": ':.2f'
    },
    height=500
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white"
)

fig.show()

In [0]:
entidades_por_bloque = (
    df_entidades_limpio
    .groupby(["obra", "titulo", "bloque_id", "tipo"])
    .size()
    .reset_index(name="cantidad")
    .sort_values(["obra", "bloque_id", "cantidad"], ascending=[True, True, False])
)

display(entidades_por_bloque.head(50))

obra,titulo,bloque_id,tipo,cantidad
CELC_MVLL,Conversación en La Catedral,1,PER,35
CELC_MVLL,Conversación en La Catedral,1,LOC,15
CELC_MVLL,Conversación en La Catedral,1,MISC,3
CELC_MVLL,Conversación en La Catedral,1,ORG,1
CELC_MVLL,Conversación en La Catedral,2,PER,14
CELC_MVLL,Conversación en La Catedral,2,LOC,5
CELC_MVLL,Conversación en La Catedral,2,MISC,4
CELC_MVLL,Conversación en La Catedral,2,ORG,1
CELC_MVLL,Conversación en La Catedral,3,LOC,21
CELC_MVLL,Conversación en La Catedral,3,PER,17


In [0]:
densidad_entidades_bloque = (
    df_entidades_limpio
    .groupby(["obra", "titulo", "bloque_id"])
    .size()
    .reset_index(name="total_entidades")
)

df_bloques_palabras = df_bloques.copy()

df_bloques_palabras["total_palabras"] = (
    df_bloques_palabras["texto_bloque"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

densidad_entidades_bloque = densidad_entidades_bloque.merge(
    df_bloques_palabras[["obra", "titulo", "bloque_id", "total_palabras"]],
    on=["obra", "titulo", "bloque_id"],
    how="left"
)

densidad_entidades_bloque["entidades_por_1000_palabras"] = (
    densidad_entidades_bloque["total_entidades"] 
    / densidad_entidades_bloque["total_palabras"] * 1000
).round(2)

display(densidad_entidades_bloque.head(30))

obra,titulo,bloque_id,total_entidades,total_palabras,entidades_por_1000_palabras
CELC_MVLL,Conversación en La Catedral,1,54,688,78.49
CELC_MVLL,Conversación en La Catedral,2,24,241,99.59
CELC_MVLL,Conversación en La Catedral,3,40,693,57.72
CELC_MVLL,Conversación en La Catedral,4,26,555,46.85
CELC_MVLL,Conversación en La Catedral,5,20,698,28.65
CELC_MVLL,Conversación en La Catedral,6,25,699,35.77
CELC_MVLL,Conversación en La Catedral,7,25,490,51.02
CELC_MVLL,Conversación en La Catedral,8,25,868,28.8
CELC_MVLL,Conversación en La Catedral,9,20,698,28.65
CELC_MVLL,Conversación en La Catedral,10,33,647,51.0


$$
densidad \ de \ entidades = \frac{cantidad \ de \ entidades} {cantidad \ de \ palabras \ del \ bloque}
$$

In [0]:
bloques_mas_entidades = (
    densidad_entidades_bloque
    .sort_values(["obra", "total_entidades"], ascending=[True, False])
    .groupby("obra", group_keys=False)
    .head(10)
)

display(bloques_mas_entidades)

obra,titulo,bloque_id,total_entidades,total_palabras,entidades_por_1000_palabras
CELC_MVLL,Conversación en La Catedral,320,81,682,118.77
CELC_MVLL,Conversación en La Catedral,213,79,693,114.0
CELC_MVLL,Conversación en La Catedral,149,78,749,104.14
CELC_MVLL,Conversación en La Catedral,281,72,914,78.77
CELC_MVLL,Conversación en La Catedral,310,71,693,102.45
CELC_MVLL,Conversación en La Catedral,311,70,674,103.86
CELC_MVLL,Conversación en La Catedral,308,68,692,98.27
CELC_MVLL,Conversación en La Catedral,319,68,667,101.95
CELC_MVLL,Conversación en La Catedral,204,67,689,97.24
CELC_MVLL,Conversación en La Catedral,32,66,976,67.62


In [0]:
bloques_mayor_densidad = (
    densidad_entidades_bloque
    .sort_values(["obra", "entidades_por_1000_palabras"], ascending=[True, False])
    .groupby("obra", group_keys=False)
    .head(10)
)

display(bloques_mayor_densidad)

obra,titulo,bloque_id,total_entidades,total_palabras,entidades_por_1000_palabras
CELC_MVLL,Conversación en La Catedral,323,54,420,128.57
CELC_MVLL,Conversación en La Catedral,320,81,682,118.77
CELC_MVLL,Conversación en La Catedral,213,79,693,114.0
CELC_MVLL,Conversación en La Catedral,146,25,239,104.6
CELC_MVLL,Conversación en La Catedral,149,78,749,104.14
CELC_MVLL,Conversación en La Catedral,311,70,674,103.86
CELC_MVLL,Conversación en La Catedral,310,71,693,102.45
CELC_MVLL,Conversación en La Catedral,319,68,667,101.95
CELC_MVLL,Conversación en La Catedral,312,66,662,99.7
CELC_MVLL,Conversación en La Catedral,2,24,241,99.59


La densidad por cada 1000 palabras permite comparar bloques de distinto tamaño. Un bloque con mayor densidad concentra más entidades en relación con su extensión.

In [0]:
for obra in densidad_entidades_bloque["obra"].unique():

    data = densidad_entidades_bloque[
        densidad_entidades_bloque["obra"] == obra
    ]

    fig = px.line(
        data,
        x="bloque_id",
        y="entidades_por_1000_palabras",
        markers=True,
        title=f"Densidad de entidades por bloque - {obra}",
        labels={
            "bloque_id": "Bloque",
            "entidades_por_1000_palabras": "Entidades por 1000 palabras"
        },
        height=500
    )

    # Hover personalizado
    fig.update_traces(
        hovertemplate=
        "<b>Bloque:</b> %{x}<br>" +
        "<b>Densidad:</b> %{y:.2f}<extra></extra>"
    )

    # Estética
    fig.update_layout(
        title_x=0.5,
        template="plotly_white"
    )

    fig.show()

# 13. Análisis de personajes

Para aproximar la extracción de personajes, se filtran las entidades de tipo `PER`. En textos literarios este tipo puede incluir personajes ficticios, autores, figuras históricas o personas mencionadas dentro del relato.

El análisis considera frecuencia total, score promedio del modelo y cantidad de bloques distintos donde aparece cada entidad.

In [0]:
# si no se trato antes en la normalizacion, seria conveniente descomentar esta linea, pero como ya lo hicimos no es necesario, de hecho podría perjudicar al analisis
# df_entidades_limpio["entidad_canonica"] = df_entidades_limpio["entidad_canonica"].replace(
#     diccionario_alias_personajes
# )

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:725)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:443)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:443)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:503)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:794)
	at com.data

In [0]:
df_personajes = df_entidades_limpio[
    df_entidades_limpio["tipo"] == "PER"
].copy()

display(df_personajes.head(10))

obra,titulo,autor,bloque_id,entidad,tipo,score,entidad_norm,entidad_canonica
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Mario Vargas Llosa,PER,0.9998625914255778,mario vargas llosa,Mario Vargas Llosa
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Santiago,PER,0.998166024684906,santiago,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Wilson,PER,0.9970278143882751,wilson,Wilson
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Zavalita,PER,0.9991453886032104,zavalita,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Carlitos,PER,0.9990795850753784,carlitos,Carlitos
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Norwin,PER,0.9999306201934814,norwin,Norwin
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Zavalita,PER,0.999855637550354,zavalita,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Santiago,PER,0.999618411064148,santiago,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Norwin,PER,0.9999040365219116,norwin,Norwin
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Director,PER,0.9996047616004944,director,Director


In [0]:
personajes_por_obra = (
    df_personajes
    .groupby(["obra", "entidad_canonica"])
    .agg(
        frecuencia=("entidad_canonica", "count"),
        score_promedio=("score", "mean"),
        bloques_distintos=("bloque_id", "nunique")
    )
    .reset_index()
    .sort_values(["obra", "frecuencia"], ascending=[True, False])
)

personajes_por_obra["score_promedio"] = personajes_por_obra["score_promedio"].round(4)

display(personajes_por_obra.head(50))

obra,entidad_canonica,frecuencia,score_promedio,bloques_distintos
CELC_MVLL,Santiago Zavala,1154,0.9986,177
CELC_MVLL,Ambrosio Pardo,799,0.9993,198
CELC_MVLL,Cayo Bermúdez,664,0.959,149
CELC_MVLL,Amalia Landa,631,0.9928,96
CELC_MVLL,Queta,407,0.987,80
CELC_MVLL,Fermín Zavala,354,0.8453,104
CELC_MVLL,Ludovico,321,0.9992,71
CELC_MVLL,Carlitos,271,0.9982,72
CELC_MVLL,Chispas Zavala,234,0.9998,64
CELC_MVLL,Popeye Arévalo,213,0.9961,54


In [0]:
top_personajes_por_obra = (
    personajes_por_obra
    .groupby("obra", group_keys=False)
    .head(20)
)

display(top_personajes_por_obra)

obra,entidad_canonica,frecuencia,score_promedio,bloques_distintos
CELC_MVLL,Santiago Zavala,1154,0.9986,177
CELC_MVLL,Ambrosio Pardo,799,0.9993,198
CELC_MVLL,Cayo Bermúdez,664,0.959,149
CELC_MVLL,Amalia Landa,631,0.9928,96
CELC_MVLL,Queta,407,0.987,80
CELC_MVLL,Fermín Zavala,354,0.8453,104
CELC_MVLL,Ludovico,321,0.9992,71
CELC_MVLL,Carlitos,271,0.9982,72
CELC_MVLL,Chispas Zavala,234,0.9998,64
CELC_MVLL,Popeye Arévalo,213,0.9961,54


In [0]:
personajes_por_bloque = (
    df_personajes
    .groupby(["obra", "titulo", "bloque_id"])
    .size()
    .reset_index(name="cantidad_personajes")
    .sort_values(["obra", "bloque_id"])
)

display(personajes_por_bloque.head(50))

obra,titulo,bloque_id,cantidad_personajes
CELC_MVLL,Conversación en La Catedral,1,35
CELC_MVLL,Conversación en La Catedral,2,14
CELC_MVLL,Conversación en La Catedral,3,17
CELC_MVLL,Conversación en La Catedral,4,8
CELC_MVLL,Conversación en La Catedral,5,16
CELC_MVLL,Conversación en La Catedral,6,17
CELC_MVLL,Conversación en La Catedral,7,18
CELC_MVLL,Conversación en La Catedral,8,20
CELC_MVLL,Conversación en La Catedral,9,15
CELC_MVLL,Conversación en La Catedral,10,21


In [0]:
personajes_por_obra["densidad_narrativa"] = (
    personajes_por_obra["frecuencia"] /
    personajes_por_obra["bloques_distintos"]
).round(2)

for obra in personajes_por_obra["obra"].unique():

    data = (
        personajes_por_obra[
            personajes_por_obra["obra"] == obra
        ]
        .head(15)
        .sort_values("frecuencia", ascending=True)
    )

    fig = px.bar(
        data,
        x="frecuencia",
        y="entidad_canonica",
        orientation="h",
        text="frecuencia",
        title=f"Personajes o personas más mencionadas - {obra}",
        labels={
            "frecuencia": "Frecuencia",
            "entidad_canonica": "Personaje"
        },
        hover_data={
            "score_promedio": ':.4f',
            "bloques_distintos": True,
            "densidad_narrativa": ':.2f'
        },
        height=600
    )

    # Hover personalizado
    fig.update_traces(
        textposition="outside",
        hovertemplate=
        "<b>Personaje:</b> %{y}<br>" +
        "<b>Frecuencia:</b> %{x}<br>" +
        "<b>Score promedio:</b> %{customdata[0]:.4f}<br>" +
        "<b>Bloques distintos:</b> %{customdata[1]}<br>" +
        "<b>Densidad narrativa:</b> %{customdata[2]:.2f}<extra></extra>"
    )

    # Diseño visual
    fig.update_layout(
        title_x=0.5,
        template="plotly_white",
        yaxis=dict(categoryorder="total ascending")
    )

    fig.show()

# 14. Análisis de lugares y escenarios

Para aproximar la extracción de escenarios, se filtran las entidades de tipo `LOC`. En este contexto, los lugares permiten identificar espacios geográficos, escenarios narrativos o referencias territoriales presentes en cada obra.

Al igual que en personajes, se calculan frecuencia, score promedio y presencia en bloques distintos.

In [0]:
df_lugares = df_entidades_limpio[
    df_entidades_limpio["tipo"] == "LOC"
].copy()

display(df_lugares.head(10))

obra,titulo,autor,bloque_id,entidad,tipo,score,entidad_norm,entidad_canonica
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Catedral,LOC,0.9997674822807312,la catedral,La Catedral
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Crónica,LOC,0.9999179244041443,la crónica,La Crónica
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,avenida Tacna,LOC,0.9997281134128571,avenida tacna,avenida Tacna
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Perú,LOC,0.999950647354126,perú,Perú
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Colmena,LOC,0.9996123909950256,colmena,Colmena
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Plaza San Martín,LOC,0.9999350309371948,plaza san martín,Plaza San Martín
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Perú,LOC,0.9994574189186096,perú,Perú
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Hotel Crillón,LOC,0.9999241828918457,hotel crillón,Hotel Crillón
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Perú,LOC,0.9987001419067383,perú,Perú
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Miraflores,LOC,0.9999513626098633,miraflores,Miraflores


In [0]:
display(df_lugares[df_lugares['obra']=='CRDLI_IGDLV'].head(10))

obra,titulo,autor,bloque_id,entidad,tipo,score,entidad_norm,entidad_canonica
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,Perú,LOC,0.9999271631240845,perú,Perú
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,Victoria,LOC,0.6841578483581543,victoria,Victoria
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,Cozco,LOC,0.9997289776802063,cozco,Cozco
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,Charcas,LOC,0.7866821885108948,charcas,Charcas
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,Chichas,LOC,0.9985367059707642,chichas,Chichas
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,1,cabo de Pasau,LOC,0.9043872356414795,cabo de pasau,cabo de Pasau
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,2,Perú,LOC,0.9999754428863525,perú,Perú
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,2,Perú,LOC,0.9999603033065796,perú,Perú
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,3,Perú,LOC,0.9999743700027466,perú,Perú
CRDLI_IGDLV,Comentarios reales de los incas,Inca Garcilaso de la Vega,3,Huelva,LOC,0.9998886585235596,huelva,Huelva


In [0]:
lugares_por_obra = (
    df_lugares
    .groupby(["obra", "entidad_norm"])
    .agg(
        frecuencia=("entidad_norm", "count"),
        score_promedio=("score", "mean"),
        bloques_distintos=("bloque_id", "nunique")
    )
    .reset_index()
    .sort_values(["obra", "frecuencia"], ascending=[True, False])
)

lugares_por_obra["score_promedio"] = lugares_por_obra["score_promedio"].round(4)

display(lugares_por_obra.head(10))

obra,entidad_norm,frecuencia,score_promedio,bloques_distintos
CELC_MVLL,lima,107,0.9998,77
CELC_MVLL,arequipa,58,0.9982,31
CELC_MVLL,perú,49,0.9997,39
CELC_MVLL,pucallpa,49,0.9909,27
CELC_MVLL,san marcos,46,0.9704,26
CELC_MVLL,chincha,37,0.9994,20
CELC_MVLL,san miguel,27,0.993,25
CELC_MVLL,miraflores,26,0.9998,25
CELC_MVLL,ancón,24,0.983,11
CELC_MVLL,ica,23,0.9996,17


In [0]:
display(lugares_por_obra[lugares_por_obra['obra']=='CRDLI_IGDLV'].head(10))

obra,entidad_norm,frecuencia,score_promedio,bloques_distintos
CRDLI_IGDLV,perú,363,0.9968,188
CRDLI_IGDLV,cozco,354,0.9915,191
CRDLI_IGDLV,españa,163,0.9992,90
CRDLI_IGDLV,chili,54,0.9903,26
CRDLI_IGDLV,quitu,53,0.9999,32
CRDLI_IGDLV,antis,31,0.9211,23
CRDLI_IGDLV,indias,29,0.993,21
CRDLI_IGDLV,collasuyu,27,0.979,20
CRDLI_IGDLV,chincha,22,0.972,8
CRDLI_IGDLV,pachacámac,21,0.994,9


In [0]:
top_lugares_por_obra = (
    lugares_por_obra
    .groupby("obra", group_keys=False)
    .head(20)
)

display(top_lugares_por_obra)

obra,entidad_norm,frecuencia,score_promedio,bloques_distintos
CELC_MVLL,lima,107,0.9998,77
CELC_MVLL,arequipa,58,0.9982,31
CELC_MVLL,perú,49,0.9997,39
CELC_MVLL,pucallpa,49,0.9909,27
CELC_MVLL,san marcos,46,0.9704,26
CELC_MVLL,chincha,37,0.9994,20
CELC_MVLL,san miguel,27,0.993,25
CELC_MVLL,miraflores,26,0.9998,25
CELC_MVLL,ancón,24,0.983,11
CELC_MVLL,ica,23,0.9996,17


In [0]:
lugares_por_bloque = (
    df_lugares
    .groupby(["obra", "titulo", "bloque_id"])
    .size()
    .reset_index(name="cantidad_lugares")
    .sort_values(["obra", "bloque_id"])
)

display(lugares_por_bloque.head(50))

obra,titulo,bloque_id,cantidad_lugares
CELC_MVLL,Conversación en La Catedral,1,15
CELC_MVLL,Conversación en La Catedral,2,5
CELC_MVLL,Conversación en La Catedral,3,21
CELC_MVLL,Conversación en La Catedral,4,15
CELC_MVLL,Conversación en La Catedral,5,1
CELC_MVLL,Conversación en La Catedral,6,4
CELC_MVLL,Conversación en La Catedral,7,4
CELC_MVLL,Conversación en La Catedral,8,3
CELC_MVLL,Conversación en La Catedral,9,4
CELC_MVLL,Conversación en La Catedral,10,10


In [0]:
for obra in lugares_por_obra["obra"].unique():

    data = (
        lugares_por_obra[
            lugares_por_obra["obra"] == obra
        ]
        .head(15)
        .sort_values("frecuencia", ascending=True)
    )

    fig = px.bar(
        data,
        x="frecuencia",
        y="entidad_norm",
        orientation="h",
        text="frecuencia",
        title=f"Lugares o escenarios más mencionados - {obra}",
        labels={
            "frecuencia": "Frecuencia",
            "entidad_norm": "Lugar o escenario"
        },
        hover_data={
            "score_promedio": ':.4f',
            "bloques_distintos": True
        },
        height=600
    )

    # Hover personalizado
    fig.update_traces(
        textposition="outside",
        hovertemplate=
        "<b>Lugar:</b> %{y}<br>" +
        "<b>Frecuencia:</b> %{x}<br>" +
        "<b>Score promedio:</b> %{customdata[0]:.4f}<br>" +
        "<b>Bloques distintos:</b> %{customdata[1]}<extra></extra>"
    )

    # Diseño visual
    fig.update_layout(
        title_x=0.5,
        template="plotly_white",
        yaxis=dict(categoryorder="total ascending")
    )

    fig.show()

# 15. Construcción de redes de coocurrencia entre personajes

La red de coocurrencia conecta dos personajes cuando aparecen dentro del mismo bloque de texto. Cada nodo representa un personaje y cada arista representa una relación de aparición conjunta.

El peso de la arista indica cuántas veces apareció ese par de personajes en los mismos bloques. Para reducir ruido, se filtran relaciones con peso menor al umbral definido en `MIN_PESO_RED`.

In [0]:
df_red_personajes = df_personajes[
    ["obra", "titulo", "bloque_id", "entidad_canonica"]
].drop_duplicates()

display(df_red_personajes.head())

obra,titulo,bloque_id,entidad_canonica
CELC_MVLL,Conversación en La Catedral,1,Mario Vargas Llosa
CELC_MVLL,Conversación en La Catedral,1,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,1,Wilson
CELC_MVLL,Conversación en La Catedral,1,Carlitos
CELC_MVLL,Conversación en La Catedral,1,Norwin


In [0]:
personajes_agrupados_bloque = (
    df_red_personajes
    .groupby(["obra", "titulo", "bloque_id"])["entidad_canonica"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="personajes")
)

personajes_agrupados_bloque["cantidad_personajes"] = personajes_agrupados_bloque["personajes"].apply(len)

display(personajes_agrupados_bloque.head(10))

obra,titulo,bloque_id,personajes,cantidad_personajes
CELC_MVLL,Conversación en La Catedral,1,"List(Carlitos, Convéncete, Director, Mario Vargas Llosa, Norwin, Orgambide, Santiago Zavala, Wilson)",8
CELC_MVLL,Conversación en La Catedral,2,"List(Ana, Becerrita, Jesús Vásquez, Milton, Norwin, Santiago Zavala)",6
CELC_MVLL,Conversación en La Catedral,3,"List(Ana, Batuquito, Director, Huxley, Popeye Arévalo, Rififí, Río Grande, Santiago Zavala, Teté Zavala)",9
CELC_MVLL,Conversación en La Catedral,4,"List(Ana, Batuque, Santiago Zavala, Solórzano)",4
CELC_MVLL,Conversación en La Catedral,5,"List(Batuque, Batuquito, Pancras, Santiago Zavala)",4
CELC_MVLL,Conversación en La Catedral,6,"List(Ambrosio Pardo, Céspedes, Fermín Zavala, Pancras, Santiago Zavala, Zoila, don, señora)",8
CELC_MVLL,Conversación en La Catedral,7,"List(Ambrosio Pardo, Chispas Zavala, Santiago Zavala, Saturnina, Teté Zavala)",5
CELC_MVLL,Conversación en La Catedral,8,"List(Ambrosio Pardo, Batuque, Carlitos, Huxley, Santiago Zavala, Saturnina, Trastabillea)",7
CELC_MVLL,Conversación en La Catedral,9,"List(Ambrosio Pardo, Batuque, Musa, Santiago Zavala)",4
CELC_MVLL,Conversación en La Catedral,10,"List(Ana, Batuque, Batuquito, Carlitos, Huxley, Norwin, Santiago Zavala)",7


In [0]:
bloques_con_coocurrencia = personajes_agrupados_bloque[
    personajes_agrupados_bloque["cantidad_personajes"] >= 2
].copy()

display(bloques_con_coocurrencia.head())

obra,titulo,bloque_id,personajes,cantidad_personajes
CELC_MVLL,Conversación en La Catedral,1,"List(Carlitos, Convéncete, Director, Mario Vargas Llosa, Norwin, Orgambide, Santiago Zavala, Wilson)",8
CELC_MVLL,Conversación en La Catedral,2,"List(Ana, Becerrita, Jesús Vásquez, Milton, Norwin, Santiago Zavala)",6
CELC_MVLL,Conversación en La Catedral,3,"List(Ana, Batuquito, Director, Huxley, Popeye Arévalo, Rififí, Río Grande, Santiago Zavala, Teté Zavala)",9
CELC_MVLL,Conversación en La Catedral,4,"List(Ana, Batuque, Santiago Zavala, Solórzano)",4
CELC_MVLL,Conversación en La Catedral,5,"List(Batuque, Batuquito, Pancras, Santiago Zavala)",4


In [0]:
from itertools import combinations

registros_coocurrencia = []

for _, row in bloques_con_coocurrencia.iterrows():
    obra = row["obra"]
    titulo = row["titulo"]
    bloque_id = row["bloque_id"]
    personajes = row["personajes"]

    for p1, p2 in combinations(personajes, 2):
        registros_coocurrencia.append({
            "obra": obra,
            "titulo": titulo,
            "bloque_id": bloque_id,
            "personaje_1": p1,
            "personaje_2": p2
        })

df_coocurrencia = pd.DataFrame(registros_coocurrencia)

display(df_coocurrencia.head(50))

obra,titulo,bloque_id,personaje_1,personaje_2
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Convéncete
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Director
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Mario Vargas Llosa
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Norwin
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Orgambide
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,1,Carlitos,Wilson
CELC_MVLL,Conversación en La Catedral,1,Convéncete,Director
CELC_MVLL,Conversación en La Catedral,1,Convéncete,Mario Vargas Llosa
CELC_MVLL,Conversación en La Catedral,1,Convéncete,Norwin


In [0]:
if len(df_coocurrencia) > 0:
    red_coocurrencia = (
        df_coocurrencia
        .groupby(["obra", "personaje_1", "personaje_2"])
        .size()
        .reset_index(name="peso")
        .sort_values(["obra", "peso"], ascending=[True, False])
    )

    display(red_coocurrencia.head(50))
else:
    print("No se encontraron coocurrencias entre personajes.")
    red_coocurrencia = pd.DataFrame(
        columns=["obra", "personaje_1", "personaje_2", "peso"]
    )

obra,personaje_1,personaje_2,peso
CELC_MVLL,Ambrosio Pardo,Santiago Zavala,104
CELC_MVLL,Ambrosio Pardo,Cayo Bermúdez,83
CELC_MVLL,Ambrosio Pardo,Fermín Zavala,76
CELC_MVLL,Carlitos,Santiago Zavala,72
CELC_MVLL,Amalia Landa,Ambrosio Pardo,70
CELC_MVLL,Fermín Zavala,Santiago Zavala,61
CELC_MVLL,Chispas Zavala,Santiago Zavala,60
CELC_MVLL,Santiago Zavala,Teté Zavala,60
CELC_MVLL,Ambrosio Pardo,don,59
CELC_MVLL,Fermín Zavala,don,58


In [0]:
red_coocurrencia_filtrada = red_coocurrencia[
    red_coocurrencia["peso"] >= MIN_PESO_RED
].copy()

display(red_coocurrencia_filtrada.head(50))

obra,personaje_1,personaje_2,peso
CELC_MVLL,Ambrosio Pardo,Santiago Zavala,104
CELC_MVLL,Ambrosio Pardo,Cayo Bermúdez,83
CELC_MVLL,Ambrosio Pardo,Fermín Zavala,76
CELC_MVLL,Carlitos,Santiago Zavala,72
CELC_MVLL,Amalia Landa,Ambrosio Pardo,70
CELC_MVLL,Fermín Zavala,Santiago Zavala,61
CELC_MVLL,Chispas Zavala,Santiago Zavala,60
CELC_MVLL,Santiago Zavala,Teté Zavala,60
CELC_MVLL,Ambrosio Pardo,don,59
CELC_MVLL,Fermín Zavala,don,58


In [0]:
grafos_por_obra = {}

for obra in red_coocurrencia_filtrada["obra"].unique():
    data_obra = red_coocurrencia_filtrada[
        red_coocurrencia_filtrada["obra"] == obra
    ]

    G = nx.Graph()

    for _, row in data_obra.iterrows():
        G.add_edge(
            row["personaje_1"],
            row["personaje_2"],
            weight=row["peso"],
            distance=1/row["peso"]
        )

    grafos_por_obra[obra] = G

    print(f"Obra: {obra}")
    print("Nodos:", G.number_of_nodes())
    print("Aristas:", G.number_of_edges())
    print("-" * 50)

Obra: CELC_MVLL
Nodos: 206
Aristas: 2103
--------------------------------------------------
Obra: CRDLI_IGDLV
Nodos: 174
Aristas: 715
--------------------------------------------------


# 16. Métricas de centralidad de la red

Una vez construida la red, se calculan métricas para identificar personajes estructuralmente importantes:

- **Grado de centralidad:** mide cuántas conexiones tiene un personaje respecto al total posible.
- **Intermediación:** identifica personajes que funcionan como puentes dentro de la red.
- **Cercanía:** estima qué tan cerca está un personaje del resto de la red.
- **Peso total de relaciones:** suma la intensidad de todas sus coocurrencias.

In [0]:
registros_centralidad = []

for obra, G in grafos_por_obra.items():

    grado_centralidad = nx.degree_centrality(G)
    intermediacion = nx.betweenness_centrality(G, weight="weight")
    cercania = nx.closeness_centrality(G, distance="distance")

    for nodo in G.nodes():
        registros_centralidad.append({
            "obra": obra,
            "personaje": nodo,
            "grado_centralidad": grado_centralidad.get(nodo, 0),
            "intermediacion": intermediacion.get(nodo, 0),
            "cercania": cercania.get(nodo, 0),
            "grado": G.degree(nodo),
            "peso_total_relaciones": sum(
                data["weight"] for _, _, data in G.edges(nodo, data=True)
            )
        })

df_centralidad = pd.DataFrame(registros_centralidad)

df_centralidad["grado_centralidad"] = df_centralidad["grado_centralidad"].round(4)
df_centralidad["intermediacion"] = df_centralidad["intermediacion"].round(4)
df_centralidad["cercania"] = df_centralidad["cercania"].round(4)

display(df_centralidad.head(50))

obra,personaje,grado_centralidad,intermediacion,cercania,grado,peso_total_relaciones
CELC_MVLL,Ambrosio Pardo,0.761,0.1122,3.861,156,1637
CELC_MVLL,Santiago Zavala,0.639,0.099,3.8188,131,1359
CELC_MVLL,Cayo Bermúdez,0.6585,0.0875,3.8262,135,1360
CELC_MVLL,Fermín Zavala,0.5073,0.0416,3.7434,104,977
CELC_MVLL,Carlitos,0.3561,0.0379,3.6376,73,551
CELC_MVLL,Amalia Landa,0.4293,0.0792,3.7124,88,727
CELC_MVLL,Chispas Zavala,0.2732,0.04,3.5974,56,455
CELC_MVLL,Teté Zavala,0.2732,0.0248,3.5975,56,470
CELC_MVLL,don,0.4439,0.0317,3.6611,91,726
CELC_MVLL,Ludovico,0.4098,0.0436,3.6776,84,704


In [0]:
top_grado_centralidad = (
    df_centralidad
    .sort_values(["obra", "grado_centralidad"], ascending=[True, False])
    .groupby("obra", group_keys=False)
    .head(10)
)

display(top_grado_centralidad)

obra,personaje,grado_centralidad,intermediacion,cercania,grado,peso_total_relaciones
CELC_MVLL,Ambrosio Pardo,0.761,0.1122,3.861,156,1637
CELC_MVLL,Cayo Bermúdez,0.6585,0.0875,3.8262,135,1360
CELC_MVLL,Santiago Zavala,0.639,0.099,3.8188,131,1359
CELC_MVLL,Fermín Zavala,0.5073,0.0416,3.7434,104,977
CELC_MVLL,don,0.4439,0.0317,3.6611,91,726
CELC_MVLL,Amalia Landa,0.4293,0.0792,3.7124,88,727
CELC_MVLL,Ludovico,0.4098,0.0436,3.6776,84,704
CELC_MVLL,Carlitos,0.3561,0.0379,3.6376,73,551
CELC_MVLL,Coronel Lozano,0.3463,0.0251,3.5321,71,467
CELC_MVLL,Presidente,0.3463,0.0253,3.537,71,462


In [0]:
top_intermediacion = (
    df_centralidad
    .sort_values(["obra", "intermediacion"], ascending=[True, False])
    .groupby("obra", group_keys=False)
    .head(10)
)

display(top_intermediacion)

obra,personaje,grado_centralidad,intermediacion,cercania,grado,peso_total_relaciones
CELC_MVLL,Ambrosio Pardo,0.761,0.1122,3.861,156,1637
CELC_MVLL,Santiago Zavala,0.639,0.099,3.8188,131,1359
CELC_MVLL,Cayo Bermúdez,0.6585,0.0875,3.8262,135,1360
CELC_MVLL,Amalia Landa,0.4293,0.0792,3.7124,88,727
CELC_MVLL,Ludovico,0.4098,0.0436,3.6776,84,704
CELC_MVLL,Fermín Zavala,0.5073,0.0416,3.7434,104,977
CELC_MVLL,Chispas Zavala,0.2732,0.04,3.5974,56,455
CELC_MVLL,Carlitos,0.3561,0.0379,3.6376,73,551
CELC_MVLL,Odría,0.3463,0.0352,3.5654,71,507
CELC_MVLL,don,0.4439,0.0317,3.6611,91,726


In [0]:
top_peso_relaciones = (
    df_centralidad
    .sort_values(["obra", "peso_total_relaciones"], ascending=[True, False])
    .groupby("obra", group_keys=False)
    .head(10)
)

display(top_peso_relaciones)

obra,personaje,grado_centralidad,intermediacion,cercania,grado,peso_total_relaciones
CELC_MVLL,Ambrosio Pardo,0.761,0.1122,3.861,156,1637
CELC_MVLL,Cayo Bermúdez,0.6585,0.0875,3.8262,135,1360
CELC_MVLL,Santiago Zavala,0.639,0.099,3.8188,131,1359
CELC_MVLL,Fermín Zavala,0.5073,0.0416,3.7434,104,977
CELC_MVLL,Amalia Landa,0.4293,0.0792,3.7124,88,727
CELC_MVLL,don,0.4439,0.0317,3.6611,91,726
CELC_MVLL,Ludovico,0.4098,0.0436,3.6776,84,704
CELC_MVLL,Queta,0.3268,0.0286,3.638,67,557
CELC_MVLL,Carlitos,0.3561,0.0379,3.6376,73,551
CELC_MVLL,Odría,0.3463,0.0352,3.5654,71,507


In [0]:
for obra in df_centralidad["obra"].unique():

    data = (
        df_centralidad[
            df_centralidad["obra"] == obra
        ]
        .sort_values("grado_centralidad", ascending=False)
        .head(15)
        .sort_values("grado_centralidad", ascending=True)
    )

    fig = px.bar(
        data,
        x="grado_centralidad",
        y="personaje",
        orientation="h",
        text="grado_centralidad",
        title=f"Personajes con mayor grado de centralidad - {obra}",
        labels={
            "grado_centralidad": "Grado de centralidad",
            "personaje": "Personaje"
        },
        hover_data={
            "intermediacion": ':.4f',
            "cercania": ':.4f',
            "grado": True,
            "peso_total_relaciones": ':.2f'
        },
        height=650
    )

    fig.update_traces(
        texttemplate="%{text:.4f}",
        textposition="outside",
        hovertemplate=
        "<b>Personaje:</b> %{y}<br>" +
        "<b>Grado centralidad:</b> %{x:.4f}<br>" +
        "<b>Intermediación:</b> %{customdata[0]:.4f}<br>" +
        "<b>Cercanía:</b> %{customdata[1]:.4f}<br>" +
        "<b>Grado:</b> %{customdata[2]}<br>" +
        "<b>Peso total relaciones:</b> %{customdata[3]:.2f}<extra></extra>"
    )

    fig.update_layout(
        title_x=0.5,
        template="plotly_white",
        yaxis=dict(categoryorder="total ascending")
    )

    fig.show()

Que Conversación en La Catedral dé a `Ambrosio Pardo` como el nodo con mayor centralidad **NO** significa automáticamente que sea:

- el protagonista;
- el personaje principal;
- el más importante narrativamente.

Y de hecho, en análisis de redes narrativas esto es completamente normal.

    Importante: Las métricas de centralidad NO miden: protagonismo literario, miden: posición estructural dentro de la red.

**¿Qué significa entonces que Ambrosio Pardo tenga alta centralidad?**

Probablemente significa que:

- conecta múltiples grupos de personajes;
- aparece junto a muchos personajes distintos;
- sirve como puente narrativo;
- participa en varias tramas;
- tiene alta coocurrencia.

Y eso encaja MUCHO con la novela.


**Mario Vargas Llosa** construye una narrativa muy fragmentada y coral.

Entonces:

* Santiago Zavala puede ser el eje psicológico;
* PERO Ambrosio puede funcionar como:
    - conector social;
    - puente político;
    - articulador de recuerdos;
    - vínculo entre clases sociales;
    - nexo entre líneas narrativas.


                                                    centralidad ≠ protagonismo


# 17. Visualización interactiva con PyVis

PyVis permite exportar las redes de personajes como archivos HTML interactivos. Los HTML se guardan dentro del volumen de Databricks, en la subcarpeta `grafos`.

In [0]:
# descomentamos si no se instalo antes

#%pip install pyvis

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:725)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:443)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:443)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:503)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:794)
	at com.data

In [0]:
# Carpeta de salida para los grafos interactivos.
Path(RUTA_GRAFOS).mkdir(parents=True, exist_ok=True)

print("Carpeta de grafos creada o verificada:", RUTA_GRAFOS)

Carpeta de grafos creada o verificada: /Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/grafos


In [0]:
archivos_grafos = []

for obra, G in grafos_por_obra.items():

    net = Network(
        height="750px",
        width="100%",
        bgcolor="#ffffff",
        font_color="#000000",
        notebook=False
    )

    # Configuración física de la red para distribuir mejor los nodos.
    net.barnes_hut(
        gravity=-8000,
        central_gravity=0.3,
        spring_length=200,
        spring_strength=0.05,
        damping=0.09
    )

    centralidad_obra = df_centralidad[df_centralidad["obra"] == obra]
    centralidad_dict = dict(zip(
        centralidad_obra["personaje"],
        centralidad_obra["grado_centralidad"]
    ))

    # Los nodos más centrales se muestran con mayor tamaño.
    for nodo in G.nodes():
        centralidad = centralidad_dict.get(nodo, 0)
        tamano = 10 + centralidad * 80

        net.add_node(
            nodo,
            label=nodo,
            size=tamano,
            title=f"{nodo}<br>Centralidad: {centralidad}"
        )

    # El grosor/valor de la arista representa la cantidad de coocurrencias.
    for origen, destino, data in G.edges(data=True):
        peso = data.get("weight", 1)

        net.add_edge(
            origen,
            destino,
            value=peso,
            title=f"Coocurrencias: {peso}"
        )

    nombre_archivo = obra.lower().replace(" ", "_").replace(":", "").replace("/", "_")
    ruta_html = f"{RUTA_GRAFOS}/red_{nombre_archivo}.html"

    net.save_graph(ruta_html)

    archivos_grafos.append({
        "obra": obra,
        "archivo_html": ruta_html
    })

df_archivos_grafos = pd.DataFrame(archivos_grafos)

display(df_archivos_grafos)

obra,archivo_html
CELC_MVLL,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/grafos/red_celc_mvll.html
CRDLI_IGDLV,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/grafos/red_crdli_igdlv.html


In [0]:
# Visualización del primer grafo generado dentro del notebook.
archivo_html = df_archivos_grafos.iloc[0]["archivo_html"]

with open(archivo_html, "r", encoding="utf-8") as f:
    html_red = f.read()

displayHTML(html_red)

<!-- 
-->
 
 


 
 
 
 
 


 
 
 
 
 
 

 
 
 
 0%

In [0]:
archivo_html = df_archivos_grafos.iloc[1]["archivo_html"]

with open(archivo_html, "r", encoding="utf-8") as f:
    html_red = f.read()

displayHTML(html_red)

<!-- 
-->
 
 


 
 
 
 
 


 
 
 
 
 
 

 
 
 
 0%

In [0]:
# # Visualización de todos los grafos generados.
# for _, row in df_archivos_grafos.iterrows():
#     print("Obra:", row["obra"])

#     with open(row["archivo_html"], "r", encoding="utf-8") as f:
#         html_red = f.read()

#     displayHTML(f"<h3>{row['obra']}</h3>" + html_red)

# 18. Exportación de resultados

En esta sección se exportan las tablas principales del análisis a archivos CSV. Todas las rutas usan el volumen configurado en `RUTA_SALIDA`

In [0]:
# Se verifica la carpeta de salida configurada en la sección 5.
Path(RUTA_SALIDA).mkdir(parents=True, exist_ok=True)

print("Ruta de salida:", RUTA_SALIDA)

Ruta de salida: /Volumes/workspace/default/mi_volumen/proyecto_nlp_libros


In [0]:
df_entidades_limpio.to_csv(
    f"{RUTA_SALIDA}/entidades_limpias_completas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo exportado: entidades_limpias_completas.csv")

Archivo exportado: entidades_limpias_completas.csv


In [0]:
frecuencia_entidades_obra.to_csv(
    f"{RUTA_SALIDA}/frecuencia_entidades_obra.csv",
    index=False,
    encoding="utf-8-sig"
)

comparacion_tipos_obra.to_csv(
    f"{RUTA_SALIDA}/comparacion_tipos_obra.csv",
    index=False,
    encoding="utf-8-sig"
)

densidad_entidades_bloque.to_csv(
    f"{RUTA_SALIDA}/densidad_entidades_bloque.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos de análisis exploratorio exportados.")

Archivos de análisis exploratorio exportados.


In [0]:
personajes_por_obra.to_csv(
    f"{RUTA_SALIDA}/personajes_por_obra.csv",
    index=False,
    encoding="utf-8-sig"
)

lugares_por_obra.to_csv(
    f"{RUTA_SALIDA}/lugares_por_obra.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos de personajes y lugares exportados.")

Archivos de personajes y lugares exportados.


In [0]:
red_coocurrencia.to_csv(
    f"{RUTA_SALIDA}/red_coocurrencia_personajes.csv",
    index=False,
    encoding="utf-8-sig"
)

red_coocurrencia_filtrada.to_csv(
    f"{RUTA_SALIDA}/red_coocurrencia_personajes_filtrada.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Redes de coocurrencia exportadas.")

Redes de coocurrencia exportadas.


In [0]:
df_centralidad.to_csv(
    f"{RUTA_SALIDA}/metricas_centralidad_personajes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Métricas de centralidad exportadas.")

Métricas de centralidad exportadas.


In [0]:
# Lista de archivos exportados en la carpeta principal del proyecto.
archivos_exportados = []

for nombre in sorted(os.listdir(RUTA_SALIDA)):
    ruta = os.path.join(RUTA_SALIDA, nombre)
    archivos_exportados.append({
        "nombre": nombre,
        "ruta": ruta,
        "tipo": "carpeta" if os.path.isdir(ruta) else "archivo"
    })

display(pd.DataFrame(archivos_exportados))

nombre,ruta,tipo
comparacion_tipos_obra.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/comparacion_tipos_obra.csv,archivo
densidad_entidades_bloque.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/densidad_entidades_bloque.csv,archivo
entidades_limpias_completas.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/entidades_limpias_completas.csv,archivo
frecuencia_entidades_obra.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/frecuencia_entidades_obra.csv,archivo
grafos,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/grafos,carpeta
lugares_por_obra.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/lugares_por_obra.csv,archivo
metricas_centralidad_personajes.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/metricas_centralidad_personajes.csv,archivo
personajes_por_obra.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/personajes_por_obra.csv,archivo
red_coocurrencia_personajes.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/red_coocurrencia_personajes.csv,archivo
red_coocurrencia_personajes_filtrada.csv,/Volumes/workspace/default/mi_volumen/proyecto_nlp_libros/red_coocurrencia_personajes_filtrada.csv,archivo


In [0]:
spark_entidades = spark.createDataFrame(df_entidades_limpio)
spark_personajes = spark.createDataFrame(personajes_por_obra)
spark_lugares = spark.createDataFrame(lugares_por_obra)
spark_centralidad = spark.createDataFrame(df_centralidad)

display(spark_entidades.limit(10))

obra,titulo,autor,bloque_id,entidad,tipo,score,entidad_norm,entidad_canonica
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Catedral,LOC,0.9997674822807312,la catedral,La Catedral
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Mario Vargas Llosa,PER,0.9998625914255778,mario vargas llosa,Mario Vargas Llosa
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,La Crónica,LOC,0.9999179244041443,la crónica,La Crónica
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Santiago,PER,0.998166024684906,santiago,Santiago Zavala
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,avenida Tacna,LOC,0.9997281134128571,avenida tacna,avenida Tacna
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Perú,LOC,0.999950647354126,perú,Perú
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Wilson,PER,0.9970278143882751,wilson,Wilson
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Colmena,LOC,0.9996123909950256,colmena,Colmena
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Plaza San Martín,LOC,0.9999350309371948,plaza san martín,Plaza San Martín
CELC_MVLL,Conversación en La Catedral,Mario Vargas Llosa,1,Perú,LOC,0.9994574189186096,perú,Perú


In [0]:
spark_entidades.createOrReplaceTempView("vw_entidades_nlp")
spark_personajes.createOrReplaceTempView("vw_personajes_nlp")
spark_lugares.createOrReplaceTempView("vw_lugares_nlp")
spark_centralidad.createOrReplaceTempView("vw_centralidad_nlp")

In [0]:
%sql
SELECT 
    obra,
    tipo,
    COUNT(*) AS total_entidades
FROM vw_entidades_nlp
GROUP BY obra, tipo
ORDER BY obra, total_entidades DESC;

obra,tipo,total_entidades
CELC_MVLL,PER,10162
CELC_MVLL,LOC,1368
CELC_MVLL,ORG,624
CELC_MVLL,MISC,577
CRDLI_IGDLV,PER,3317
CRDLI_IGDLV,LOC,2698
CRDLI_IGDLV,MISC,2126
CRDLI_IGDLV,ORG,596


# 19. Interpretación de resultados

In [0]:
for obra in df_entidades_limpio["obra"].unique():
    print("=" * 90)
    print(f"OBRA: {obra}")
    print("=" * 90)

    total_entidades = len(df_entidades_limpio[df_entidades_limpio["obra"] == obra])
    total_personajes = len(df_personajes[df_personajes["obra"] == obra])
    total_lugares = len(df_lugares[df_lugares["obra"] == obra])

    print(f"Total de entidades detectadas: {total_entidades}")
    print(f"Total de menciones de personajes/personas: {total_personajes}")
    print(f"Total de menciones de lugares/escenarios: {total_lugares}")

    print("\nTop 5 personajes:")
    display(
        personajes_por_obra[personajes_por_obra["obra"] == obra]
        .head(5)
    )

    print("\nTop 5 lugares:")
    display(
        lugares_por_obra[lugares_por_obra["obra"] == obra]
        .head(5)
    )

OBRA: CELC_MVLL
Total de entidades detectadas: 12731
Total de menciones de personajes/personas: 10162
Total de menciones de lugares/escenarios: 1368

Top 5 personajes:


obra,entidad_canonica,frecuencia,score_promedio,bloques_distintos,densidad_narrativa
CELC_MVLL,Santiago Zavala,1154,0.9986,177,6.52
CELC_MVLL,Ambrosio Pardo,799,0.9993,198,4.04
CELC_MVLL,Cayo Bermúdez,664,0.959,149,4.46
CELC_MVLL,Amalia Landa,631,0.9928,96,6.57
CELC_MVLL,Queta,407,0.987,80,5.09



Top 5 lugares:


obra,entidad_norm,frecuencia,score_promedio,bloques_distintos
CELC_MVLL,lima,107,0.9998,77
CELC_MVLL,arequipa,58,0.9982,31
CELC_MVLL,perú,49,0.9997,39
CELC_MVLL,pucallpa,49,0.9909,27
CELC_MVLL,san marcos,46,0.9704,26


OBRA: CRDLI_IGDLV
Total de entidades detectadas: 8737
Total de menciones de personajes/personas: 3317
Total de menciones de lugares/escenarios: 2698

Top 5 personajes:


obra,entidad_canonica,frecuencia,score_promedio,bloques_distintos,densidad_narrativa
CRDLI_IGDLV,Inca,642,0.94,204,3.15
CRDLI_IGDLV,Rey,235,0.9728,144,1.63
CRDLI_IGDLV,Reyes,189,0.9695,121,1.56
CRDLI_IGDLV,Sol,132,0.8293,68,1.94
CRDLI_IGDLV,Incas,107,0.76,71,1.51



Top 5 lugares:


obra,entidad_norm,frecuencia,score_promedio,bloques_distintos
CRDLI_IGDLV,perú,363,0.9968,188
CRDLI_IGDLV,cozco,354,0.9915,191
CRDLI_IGDLV,españa,163,0.9992,90
CRDLI_IGDLV,chili,54,0.9903,26
CRDLI_IGDLV,quitu,53,0.9999,32


In [0]:
for obra in df_centralidad["obra"].unique():
    print("=" * 90)
    print(f"PERSONAJES CENTRALES EN: {obra}")
    print("=" * 90)

    data_obra = (
        df_centralidad[df_centralidad["obra"] == obra]
        .sort_values("grado_centralidad", ascending=False)
        .head(5)
    )

    for _, row in data_obra.iterrows():
        print(
            f"{row['personaje']} destaca por su nivel de conexión en la red, "
            f"con un grado de centralidad de {row['grado_centralidad']} "
            f"y un peso total de relaciones de {row['peso_total_relaciones']}."
        )

    print()

PERSONAJES CENTRALES EN: CELC_MVLL
Ambrosio Pardo destaca por su nivel de conexión en la red, con un grado de centralidad de 0.761 y un peso total de relaciones de 1637.
Cayo Bermúdez destaca por su nivel de conexión en la red, con un grado de centralidad de 0.6585 y un peso total de relaciones de 1360.
Santiago Zavala destaca por su nivel de conexión en la red, con un grado de centralidad de 0.639 y un peso total de relaciones de 1359.
Fermín Zavala destaca por su nivel de conexión en la red, con un grado de centralidad de 0.5073 y un peso total de relaciones de 977.
don destaca por su nivel de conexión en la red, con un grado de centralidad de 0.4439 y un peso total de relaciones de 726.

PERSONAJES CENTRALES EN: CRDLI_IGDLV
Inca destaca por su nivel de conexión en la red, con un grado de centralidad de 0.7572 y un peso total de relaciones de 937.
Rey destaca por su nivel de conexión en la red, con un grado de centralidad de 0.5896 y un peso total de relaciones de 667.
Reyes destaca 

In [0]:
for obra in lugares_por_obra["obra"].unique():
    print("=" * 90)
    print(f"LUGARES PREDOMINANTES EN: {obra}")
    print("=" * 90)

    data_obra = lugares_por_obra[
        lugares_por_obra["obra"] == obra
    ].head(5)

    for _, row in data_obra.iterrows():
        print(
            f"{row['entidad_norm']} aparece con una frecuencia de "
            f"{row['frecuencia']} menciones, distribuidas en "
            f"{row['bloques_distintos']} bloques distintos."
        )

    print()

LUGARES PREDOMINANTES EN: CELC_MVLL
lima aparece con una frecuencia de 107 menciones, distribuidas en 77 bloques distintos.
arequipa aparece con una frecuencia de 58 menciones, distribuidas en 31 bloques distintos.
perú aparece con una frecuencia de 49 menciones, distribuidas en 39 bloques distintos.
pucallpa aparece con una frecuencia de 49 menciones, distribuidas en 27 bloques distintos.
san marcos aparece con una frecuencia de 46 menciones, distribuidas en 26 bloques distintos.

LUGARES PREDOMINANTES EN: CRDLI_IGDLV
perú aparece con una frecuencia de 363 menciones, distribuidas en 188 bloques distintos.
cozco aparece con una frecuencia de 354 menciones, distribuidas en 191 bloques distintos.
españa aparece con una frecuencia de 163 menciones, distribuidas en 90 bloques distintos.
chili aparece con una frecuencia de 54 menciones, distribuidas en 26 bloques distintos.
quitu aparece con una frecuencia de 53 menciones, distribuidas en 32 bloques distintos.



In [0]:
for obra, G in grafos_por_obra.items():
    print("=" * 90)
    print(f"RED DE COOCCURRENCIA EN: {obra}")
    print("=" * 90)

    print(f"Número de personajes en la red: {G.number_of_nodes()}")
    print(f"Número de relaciones entre personajes: {G.number_of_edges()}")

    if G.number_of_nodes() > 0:
        densidad = nx.density(G)
        print(f"Densidad de la red: {round(densidad, 4)}")

    print()

RED DE COOCCURRENCIA EN: CELC_MVLL
Número de personajes en la red: 206
Número de relaciones entre personajes: 2103
Densidad de la red: 0.0996

RED DE COOCCURRENCIA EN: CRDLI_IGDLV
Número de personajes en la red: 174
Número de relaciones entre personajes: 715
Densidad de la red: 0.0475



La red de coocurrencia se construyó a partir de bloques textuales de tamaño fijo, por lo que una relación entre personajes indica proximidad textual dentro del mismo bloque, no necesariamente interacción narrativa directa. Por ello, las relaciones detectadas deben interpretarse como asociaciones aproximadas y no como vínculos confirmados entre personajes.

**Interpretación general de resultados**

El análisis de entidades nombradas permitió identificar personas, lugares, organizaciones y otras referencias relevantes dentro de las dos obras estudiadas. Las entidades clasificadas como personas fueron utilizadas como una aproximación inicial a los personajes o figuras mencionadas, mientras que las entidades de tipo lugar permitieron reconocer escenarios o espacios geográficos presentes en el corpus.

La comparación entre obras permite observar diferencias en la distribución de entidades. Una obra puede presentar mayor concentración de personajes, mientras que otra puede destacar por la presencia de lugares, referencias históricas o instituciones. Esta diferencia es importante porque refleja la naturaleza textual de cada obra: una puede tener una orientación más narrativa y otra una orientación más histórica o descriptiva.

Además, la construcción de redes de coocurrencia permitió analizar qué personajes aparecen juntos dentro de los mismos bloques textuales. A partir de estas relaciones, las métricas de centralidad ayudaron a identificar personajes con mayor peso estructural dentro de cada obra. Sin embargo, estos resultados deben interpretarse con cuidado, ya que la extracción automática de entidades puede presentar errores, especialmente en textos literarios, históricos o con formas antiguas del español.

# 20. Conclusiones

El procesamiento computacional del corpus permitió extraer y organizar información relevante sobre entidades nombradas presentes en las obras analizadas. Mediante el uso de Flair para el reconocimiento de entidades y spaCy como apoyo lingüístico, fue posible identificar menciones asociadas a personas, lugares, organizaciones y otras referencias textuales. Esta información sirvió como base para construir una lectura cuantitativa inicial de los personajes, escenarios y relaciones internas de cada obra.

El análisis de personajes mostró qué nombres aparecen con mayor frecuencia y cuáles tienen mayor presencia en distintos bloques del texto. A partir de la red de coocurrencia, también fue posible observar qué personajes tienden a aparecer juntos y cuáles ocupan posiciones más centrales dentro de la estructura textual. Las métricas de centralidad permitieron complementar la frecuencia simple, ya que un personaje no solo puede ser importante por aparecer muchas veces, sino también por conectar diferentes grupos de personajes.

El análisis de lugares y escenarios permitió reconocer los espacios más mencionados dentro de cada obra. Esto resulta especialmente útil para comparar textos literarios e históricos, ya que los lugares pueden funcionar como escenarios narrativos, referencias geográficas o marcas contextuales. En conjunto, estos resultados muestran que las técnicas de NLP pueden apoyar el análisis literario al ofrecer evidencia cuantitativa sobre patrones de aparición, distribución y relación entre entidades.

Finalmente, es importante señalar que los resultados no deben considerarse definitivos sin una revisión humana. Los modelos de reconocimiento de entidades pueden confundir nombres, lugares u organizaciones, sobre todo en textos con lenguaje literario, histórico o antiguo. Por ello, este análisis debe entenderse como una herramienta de apoyo para la interpretación, no como un reemplazo de la lectura crítica de las obras.